In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import os

print("Day 43 libraries imported successfully!")

Day 43 libraries imported successfully!


In [2]:
df = pd.read_csv("../cleaned_dataset.csv")

df["order_date"] = pd.to_datetime(df["order_date"])

print("Customer dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Customer dataset loaded successfully!
Rows: 51290
Columns: 23


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,shipping_days,profit_margin
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,"Tenex Lockers, Blue",408.0,2,0.0,106.140,35.46,Medium,2011,5,26.014706
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,"Acme Trimmer, High Speed",120.0,3,0.1,36.036,9.72,Medium,2011,7,30.030000
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,"Tenex Box, Single Width",66.0,4,0.0,29.640,8.17,High,2011,4,44.909091
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,"Enermax Note Cards, Premium",45.0,3,0.5,-26.055,4.82,High,2011,4,-57.900000
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,"Eldon Light Bulb, Duo Pack",114.0,5,0.1,37.770,4.70,Medium,2011,7,33.131579


In [3]:
customer_data = df.groupby("customer_name").agg(
    total_sales=("sales", "sum"),
    total_profit=("profit", "sum"),
    total_quantity=("quantity", "sum"),
    number_of_orders=("order_id", "nunique"),
    average_discount=("discount", "mean"),
    average_shipping_days=("shipping_days", "mean"),
    first_purchase=("order_date", "min"),
    last_purchase=("order_date", "max")
).reset_index()

print("Customer-level features created successfully!")

print("Number of customers:", len(customer_data))

display(customer_data.head())

Customer-level features created successfully!
Number of customers: 795


,customer_name,total_sales,total_profit,total_quantity,number_of_orders,average_discount,average_shipping_days,first_purchase,last_purchase
0,Aaron Bergman,24646.0,4683.20800,301,37,0.110112,3.707865,2011-02-19,2014-12-15
1,Aaron Hawkins,20759.0,2450.92904,231,34,0.160929,3.464286,2011-04-22,2014-12-19
2,Aaron Smayling,14207.0,369.16180,211,31,0.167667,4.433333,2011-03-21,2014-12-08
3,Adam Bellavance,20189.0,4979.97690,262,41,0.131765,3.514706,2011-01-07,2014-11-26
4,Adam Hart,21720.0,1902.03342,293,42,0.093238,3.654762,2011-06-10,2014-12-29


In [4]:
analysis_date = df["order_date"].max()

cutoff_date = analysis_date - pd.Timedelta(days=60)

historical_data = df[df["order_date"] <= cutoff_date]
future_data = df[df["order_date"] > cutoff_date]

future_customers = set(
    future_data["customer_name"].unique()
)

customer_data["churn_risk"] = (
    ~customer_data["customer_name"].isin(future_customers)
).astype(int)

print("Churn-risk target created successfully!")

print("Cutoff date:", cutoff_date)
print("Customers:", len(customer_data))
print("Risk customers:", customer_data["churn_risk"].sum())
print("Non-risk customers:", (customer_data["churn_risk"] == 0).sum())

display(
    customer_data[
        ["customer_name", "churn_risk"]
    ].head()
)

Churn-risk target created successfully!
Cutoff date: 2014-11-01 00:00:00
Customers: 795
Risk customers: 49
Non-risk customers: 746


C:\Users\arjun\AppData\Local\Temp\ipykernel_20356\1526503303.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  cutoff_date = analysis_date - pd.Timedelta(days=60)


,customer_name,churn_risk
0,Aaron Bergman,0
1,Aaron Hawkins,0
2,Aaron Smayling,0
3,Adam Bellavance,0
4,Adam Hart,0


In [5]:
features = [
    "total_sales",
    "total_profit",
    "total_quantity",
    "number_of_orders",
    "average_discount",
    "average_shipping_days"
]

X = customer_data[features]

y = customer_data["churn_risk"]

print("Model features prepared successfully!")

print("Input features:", X.shape)
print("Target values:", y.shape)

display(X.head())

Model features prepared successfully!
Input features: (795, 6)
Target values: (795,)


,total_sales,total_profit,total_quantity,number_of_orders,average_discount,average_shipping_days
0,24646.0,4683.20800,301,37,0.110112,3.707865
1,20759.0,2450.92904,231,34,0.160929,3.464286
2,14207.0,369.16180,211,31,0.167667,4.433333
3,20189.0,4979.97690,262,41,0.131765,3.514706
4,21720.0,1902.03342,293,42,0.093238,3.654762


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Data split successfully!")

print("Training customers:", len(X_train))
print("Testing customers:", len(X_test))

Data split successfully!
Training customers: 636
Testing customers: 159


In [7]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [8]:
y_pred = model.predict(X_test)

print("Model prediction completed!")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model prediction completed!

Classification Report:
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       149
           1       0.00      0.00      0.00        10

    accuracy                           0.94       159
   macro avg       0.47      0.50      0.48       159
weighted avg       0.88      0.94      0.91       159



C:\Users\arjun\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\arjun\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\arjun\AppData\Roaming\Python\Python314\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [9]:
model_path = "customer_risk_model.pkl"

joblib.dump(model, model_path)

print("Customer risk model saved successfully!")
print("Model file:", model_path)
print("File exists:", os.path.exists(model_path))

Customer risk model saved successfully!
Model file: customer_risk_model.pkl
File exists: True
